In [1]:

!apt-get update -qq

!pip install -q transformers==4.45.2 accelerate==0.30.1 einops==0.8.0 datasets pillow matplotlib requests


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 38.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:

import json
import math
import os

import torch
from google.colab import drive
from PIL import Image
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoProcessor


In [3]:

drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[STATUS] Hardware engine initialized: {device}")

DATASET_PATH = "/content/drive/MyDrive/Florence-2 Grasp Point Prediction Security/data/raw/metadata_eval.json"
OUTPUT_PATH = "/content/drive/MyDrive/Florence-2 Grasp Point Prediction Security/outputs/threat_eval_results.json"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

assert os.path.exists(DATASET_PATH), f"[CRITICAL] Dataset file not found at: {DATASET_PATH}"
print(f"[STATUS] Filesystem verified. Threat Matrix dataset located.")


Mounted at /content/drive
[STATUS] Hardware engine initialized: cuda
[STATUS] Filesystem verified. Threat Matrix dataset located.


In [4]:

model_id = "microsoft/Florence-2-large"
print(f"[STATUS] Initializing {model_id} with explicit FP16 precision...")

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device).eval()

print(f"[STATUS] {model_id} successfully loaded and allocated on {device} (FP16).")


[STATUS] Initializing microsoft/Florence-2-large with explicit FP16 precision...


preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

...

modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

[STATUS] microsoft/Florence-2-large successfully loaded and allocated on cuda (FP16).


In [5]:

def get_box_center(box):
    x1, y1, x2, y2 = box
    center_x = (x1 + x2) / 2.0
    center_y = (y1 + y2) / 2.0
    return (center_x, center_y)

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def inference(image, task_prompt, text_input=None):
    prompt = task_prompt if text_input is None else task_prompt + text_input
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch.float16)

    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        early_stopping=False,
        do_sample=False,
        num_beams=3,
    )

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

    parsed_answer = processor.post_process_generation(
        generated_text,
        task=task_prompt,
        image_size=(image.width, image.height)
    )

    return parsed_answer

print("\n✅[STATUS] Inference utilities and coordinate logic initialized.")



✅[STATUS] Inference utilities and coordinate logic initialized.


In [6]:

print("[INFO] Loading Threat Matrix dataset...")
with open(DATASET_PATH, "r") as f:
    eval_data = json.load(f)

results = []

for entry in tqdm(eval_data, desc="Evaluating Threat Matrix"):
    image_path = entry["image_path"]
    target_label = entry["label"]
    threat_class = entry["threat_class"]

    gt_raw = entry["points"][0]

    try:
        img = Image.open(image_path).convert("RGB")
        w, h = img.size

        gt_x_px = (gt_raw['x'] / 100.0) * w
        gt_y_px = (gt_raw['y'] / 100.0) * h

        prompt_point = entry["prompts"]["standard_point"]
        prompt_grasp = entry["prompts"]["adversarial_grasp"]

        err_point, pred_x_pt, pred_y_pt, box_point = None, None, None, None
        err_grasp, pred_x_gr, pred_y_gr, box_grasp = None, None, None, None

        raw_output_point = inference(img, "<CAPTION_TO_PHRASE_GROUNDING>", text_input=prompt_point)
        parsed_point = raw_output_point["<CAPTION_TO_PHRASE_GROUNDING>"]

        if len(parsed_point["bboxes"]) > 0:
            box_point = parsed_point["bboxes"][0]
            pred_x_pt, pred_y_pt = get_box_center(box_point)
            err_point = euclidean_distance((pred_x_pt, pred_y_pt), (gt_x_px, gt_y_px))

        raw_output_grasp = inference(img, "<CAPTION_TO_PHRASE_GROUNDING>", text_input=prompt_grasp)
        parsed_grasp = raw_output_grasp["<CAPTION_TO_PHRASE_GROUNDING>"]

        if len(parsed_grasp["bboxes"]) > 0:
            box_grasp = parsed_grasp["bboxes"][0]
            pred_x_gr, pred_y_gr = get_box_center(box_grasp)
            err_grasp = euclidean_distance((pred_x_gr, pred_y_gr), (gt_x_px, gt_y_px))

        results.append({
            "id": entry["id"],
            "label": target_label,
            "threat_class": threat_class,
            "image_width": w,
            "image_height": h,
            "gt_x": gt_x_px,
            "gt_y": gt_y_px,
            "pred_x_point": pred_x_pt,
            "pred_y_point": pred_y_pt,
            "box_point": box_point,
            "error_standard": err_point,
            "pred_x_grasp": pred_x_gr,
            "pred_y_grasp": pred_y_gr,
            "box_grasp": box_grasp,
            "error_adversarial": err_grasp
        })

    except Exception as e:
        print(f"\n[ERROR] Pipeline failure on instance {entry['id']}: {e}")
        continue

with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n[INFO] ✅ Threat Matrix Evaluation Complete! Standardized JSON report generated at: {OUTPUT_PATH}")


[INFO] Loading Threat Matrix dataset...


Evaluating Threat Matrix: 100%|██████████| 339/339 [07:37<00:00,  1.35s/it]


[INFO] ✅ Threat Matrix Evaluation Complete! Standardized JSON report generated at: /content/drive/MyDrive/Florence-2 Grasp Point Prediction Security/outputs/threat_eval_results.json
